## Post-Intervention Chain Probability

In [1]:
%load_ext autoreload
%autoreload 2

### Overview

Measures how much probability the model still assigns to its original reasoning-chain continuation versus the counterfactual one, right after a number swap, without generating anything new — purely via teacher-forced probabilities.

### Set-up

In [ ]:
import sys
sys.path.append("../../src")
sys.path.append("src")

import torch
import gc
from tqdm import tqdm

import _config
from _intervention import get_label_probability

In [3]:
prompt_config = _config.PromptConfig(
    model_type="GPT-OSS_stepwise", # GPT-OSS or R1
    prompt_type="h", # empty or pre_result or pre_sum
)
run_config = _config.RunConfig(
    experiment_root="experiments/token_intervention",
    result_dir="post_intervention_chain_probability",
    output_filename=f"{prompt_config.stem}.csv",
)
batch_size = 24

model, tokenizer = _config.load_model(prompt_config.model_type)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [4]:
divided_prompts = _config.load_divided_prompts(prompt_config)
print(f"loaded {len(divided_prompts)} divided prompts")

loaded 3072 divided prompts


In [5]:
def get_generation_clean_prompt(row):
    return row['base_before'] + str(row['base_number'])

# Batch code below uses _config.build_number_prompts for the same construction.

In [6]:
def get_generation_intervention_prompt(row):
    return row['base_before'] + str(row['source_number'])

# Batch code below uses _config.build_number_prompts for the same construction.

### Run: teacher-forced chain probabilities

For each divided prompt, teacher-forces the probability of the factual and counterfactual continuations under both the clean and the intervened prompt.

In [ ]:
# Get header of divided prompts dataset
header = list(divided_prompts.columns) + ['chain_prob', 'post_intervention_factual_chain_prob', 'post_intervention_counterfactual_chain_prob']
filepath = _config.build_run_output_filepath(prompt_config, run_config, header)

for i in tqdm(range(0, len(divided_prompts), batch_size)):
    torch.cuda.empty_cache()
    gc.collect()
    batch_rows = divided_prompts.iloc[i:i+batch_size]
    
    # Prepare batch of intervention prompts
    clean_prompts = _config.build_number_prompts(batch_rows, "base_number")
    intervention_prompts = _config.build_number_prompts(batch_rows, "source_number")
    factual_suffix = [row['base_after'] for _, row in batch_rows.iterrows()]
    counterfactual_suffix = [row['source_after'] for _, row in batch_rows.iterrows()]
    
    # Tokenize all prompts in the batch
    clean_tokens = tokenizer(clean_prompts, add_special_tokens=False, return_tensors="pt", padding=True, padding_side="left").to(model.device)
    intervention_tokens = tokenizer(intervention_prompts, add_special_tokens=False, return_tensors="pt", padding=True, padding_side="left").to(model.device)
    factual_suffix_tokens, factual_suffix_mask = _config.prepare_sequence_label_tokens(tokenizer, factual_suffix, device=model.device)
    counterfactual_suffix_tokens, counterfactual_suffix_mask = _config.prepare_sequence_label_tokens(tokenizer, counterfactual_suffix, device=model.device)
    
    # Get label probabilities
    clean_probs = get_label_probability(model, clean_tokens, factual_suffix_tokens, factual_suffix_mask)
    intervention_factual_probs = get_label_probability(model, intervention_tokens, factual_suffix_tokens, factual_suffix_mask)
    intervention_counterfactual_probs = get_label_probability(model, intervention_tokens, counterfactual_suffix_tokens, counterfactual_suffix_mask)
    
    # Process each generated text in the batch
    for j, (_, row) in enumerate(batch_rows.iterrows()):
        _config.write_to_csv(filepath, row.to_list() + [clean_probs[j].item(), intervention_factual_probs[j].item(), intervention_counterfactual_probs[j].item()])


  0%|                                                                                                                       | 0/128 [00:00<?, ?it/s]

 20%|█████████████████████▍                                                                                        | 25/128 [09:35<39:34, 23.06s/it]